# Tau-måling: Qwen2.5 familien

**Køyr celle 1 (install), deretter celle 2 (alt anna). Det er alt.**

VRAM-krav (4-bit): 7B~5GB, 14B~10GB, 32B~18GB, 72B~40GB

*Opne via: Colab → File → Open notebook → GitHub-fana → nsolland/Tofoo-*

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch numpy

In [ ]:
import torch, numpy as np, math, json
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Konstanter ---
TAU_MIN = math.exp(-0.5772156649)   # 0.5615
TAU_MAX = 1.0 / 1.2020569032        # 0.8319

TEXTS = {
    "coherent":   "The structure of a system reveals itself through the patterns it sustains over time. Coherence emerges when local rules propagate consistently across all scales.",
    "random":     "Quantum entropy bicycle seventeen. Mountain glass decides purple. The concept flows between adjacent memory structures. Floating decisions cascade.",
    "repetitive": "the " * 24,
}

MODELLAR = [
    {"key": "7B",  "name": "Qwen/Qwen2.5-7B",  "min_gb": 6},
    {"key": "14B", "name": "Qwen/Qwen2.5-14B", "min_gb": 11},
    {"key": "32B", "name": "Qwen/Qwen2.5-32B", "min_gb": 19},
    {"key": "72B", "name": "Qwen/Qwen2.5-72B", "min_gb": 42},
]

# --- VRAM ---
if not torch.cuda.is_available():
    raise RuntimeError("Ingen GPU — aktiver Runtime > Change runtime type > T4 GPU")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"Goldilocks: [{TAU_MIN:.4f}, {TAU_MAX:.4f}]")

KAN_KJØYRE = [m for m in MODELLAR if m["min_gb"] <= vram_gb]
IKKJE      = [m for m in MODELLAR if m["min_gb"] >  vram_gb]
print(f"\nKøyrer: {[m['key'] for m in KAN_KJØYRE]}")
if IKKJE:
    print(f"Hoppar over (for stor): {[m['key'] for m in IKKJE]}")

# --- Tau ---
def tau(hidden):
    H  = hidden.squeeze(0).float().cpu().numpy()
    _,s,_ = np.linalg.svd(H, full_matrices=False)
    p  = s**2 / (s**2).sum()
    p  = p[p > 1e-10]
    r_eff = np.exp(-np.sum(p * np.log(p)))
    return float(r_eff / min(H.shape))

# --- Målingar ---
alle = {}
for m in KAN_KJØYRE:
    print(f"\n{'='*55}")
    print(f"  {m['name']}")
    print(f"{'='*55}")
    try:
        tok = AutoTokenizer.from_pretrained(m["name"], trust_remote_code=True)
        # AutoModelForCausalLM + output_hidden_states i forward-kallet
        # løyser hidden_states=None med 4-bit bitsandbytes
        mdl = AutoModelForCausalLM.from_pretrained(
            m["name"], trust_remote_code=True,
            torch_dtype=torch.float16, device_map="auto", load_in_4bit=True,
        )
        mdl.eval()

        res = {}
        for label, text in TEXTS.items():
            inp = tok(text, return_tensors="pt", truncation=True, max_length=256)
            inp = {k: v.to(mdl.device) for k, v in inp.items()}
            with torch.no_grad():
                out = mdl(**inp, output_hidden_states=True)
            if out.hidden_states is None:
                print(f"  FEIL: hidden_states=None for {label} — prøver model.config")
                mdl.config.output_hidden_states = True
                out = mdl(**inp, output_hidden_states=True)
            if out.hidden_states is None:
                print(f"  FEIL: hidden_states framleis None — hoppar over {label}")
                continue
            t = tau(out.hidden_states[-1])
            res[label] = t
            sone = "GOLDILOCKS ★" if TAU_MIN <= t <= TAU_MAX else ("BELOW" if t < TAU_MIN else "ABOVE")
            print(f"  [{label:>10}]  tau={t:.4f}  {sone}")

        if len(res) == 3:
            ok = res["coherent"] > res["random"] > res["repetitive"]
            print(f"  Rekkefølge koherent>tilfeldig>repetitivt: {'JA ✓' if ok else 'NEI ✗'}")
            alle[m["key"]] = {"name": m["name"], "res": res, "ok": ok}
        else:
            print(f"  Berre {len(res)}/3 tekstar fullførte")

        del mdl
        torch.cuda.empty_cache()

    except torch.cuda.OutOfMemoryError:
        print(f"  OOM — hoppar over")
    except Exception as e:
        import traceback
        print(f"  FEIL: {e}")
        traceback.print_exc()

# --- Sammendrag ---
print(f"\n{'='*65}")
print("SAMMENDRAG")
print(f"{'='*65}")
if not alle:
    print("  Ingen modellar fullførte — sjå feilmeldingar ovanfor")
else:
    print(f"{'Modell':<8} {'tau(koh)':<12} {'tau(rand)':<12} {'tau(rep)':<12} OK?")
    print("-"*55)
    for k, d in alle.items():
        r = d["res"]
        print(f"{k:<8} {r['coherent']:<12.4f} {r['random']:<12.4f} {r['repetitive']:<12.4f} {'JA ✓' if d['ok'] else 'NEI ✗'}")

print(f"\nGoldilocks: [{TAU_MIN:.4f}, {TAU_MAX:.4f}]")
print("\nTidlegare målingar:")
for namn, t in [("GPT-2 (117M)",0.060),("gpt-neo (1.3B)",0.201),("Phi-2 (2.7B)",0.163),("Qwen2.5-7B",0.156),("Mistral-7B",0.257)]:
    print(f"  {namn:<22}  tau={t:.4f}")

# --- Last ned JSON ---
if alle:
    try:
        from google.colab import files
        fn = f"tau_qwen25_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
        with open(fn, "w") as f:
            json.dump(alle, f, indent=2)
        files.download(fn)
        print(f"\nLasta ned: {fn}")
    except:
        print(f"\nJSON:\n{json.dumps(alle, indent=2)}")